In [1]:
# MICrONS Visual Decoding with GridSearchCV

!pip install -q dandi remfile pynwb h5py scikit-learn

import numpy as np
import matplotlib.pyplot as plt
import remfile
import h5py
from pynwb import NWBHDF5IO
from dandi.dandiapi import DandiAPIClient
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ============================================================================
# Connecting to dataset
# ============================================================================

def connect_to_data():
    """Connect to the dataset and return the nwb file."""
    client = DandiAPIClient()
    dandiset = client.get_dandiset("000402", "draft")
    all_assets = dandiset.get_assets()

    asset = None
    for a in all_assets:
        if a.path.endswith(".nwb"):
            asset = a
            break

    s3_url = asset.get_content_url(follow_redirects=1, strip_query=True)
    rf = remfile.File(s3_url)
    h5 = h5py.File(rf, "r")
    io = NWBHDF5IO(file=h5, load_namespaces=True)
    nwb = io.read()

    return nwb


def get_neural_data(nwb):
    """Get the neural recording data."""
    ophys = nwb.processing["ophys"]
    fluorescence = ophys.data_interfaces["Fluorescence"]
    rs = fluorescence.roi_response_series["RoiResponseSeries3"]

    print("Total neurons:", rs.data.shape[1])
    print("Total timepoints:", rs.data.shape[0])

    return rs


def get_stimulus_info(nwb):
    """Get information about when stimuli were shown."""
    clip_intervals = nwb.intervals['Clip']

    clip_starts = np.array(clip_intervals.start_time[:])
    clip_stops = np.array(clip_intervals.stop_time[:])
    clip_types = np.array(clip_intervals.short_movie_name[:])

    print("Total stimulus presentations:", len(clip_starts))

    unique_types = np.unique(clip_types)
    for stim_type in unique_types:
        count = np.sum(clip_types == stim_type)
        print(f"  {stim_type}: {count} presentations")

    return clip_starts, clip_stops, clip_types


# ============================================================================
# Feature selection
# ============================================================================

def select_best_neurons(rs, timestamps, clip_starts, clip_stops, clip_types,
                       target_conditions, n_select=200):
    """
    Select the most informative neurons using ANOVA F-test.
    Neurons with highest F-scores best discriminate between conditions.
    """

    # First extract features for ALL neurons
    all_neurons = list(range(rs.data.shape[1]))
    X_all, y_all, _ = extract_neural_features(
        rs, timestamps, clip_starts, clip_stops, clip_types,
        all_neurons, target_conditions, window_duration=None
    )

    # Calculate ANOVA F-score for each neuron
    # Higher F-score = better discrimination between classes
    from sklearn.feature_selection import f_classif
    f_scores, p_values = f_classif(X_all, y_all)

    # Select top N neurons with highest F-scores
    best_indices = np.argsort(f_scores)[-n_select:]
    best_indices = sorted(best_indices.tolist())  # Convert to list and sort

    return best_indices


# ============================================================================
# Extractin data
# ============================================================================

def extract_neural_features(rs, timestamps, clip_starts, clip_stops, clip_types,
                           neuron_list, target_conditions, window_duration=None,
                           use_rich_features=False):
    """
    Extract neural features for decoding WITHOUT z-scoring here.

    We're transforming:
      Neural recording: [time × neurons] continuous time series
    Into:
      Feature matrix: [trials × features] for classification

    - One trial = one presentation of one stimulus
    - For each trial, we extract a summary of neural activity
    - Example: If "Cinematic" was shown 50 times, we get 50 trials

    Two modes controlled by use_rich_features flag:

    MODE 1: Simple (use_rich_features=False)
    - Just compute mean fluorescence for each neuron during stimulus
    - Features: [neuron_1_mean, neuron_2_mean, ..., neuron_N_mean]
    - Simple but loses temporal information

    MODE 2: Rich (use_rich_features=True)
    - Compute multiple statistics capturing temporal dynamics:
      * Mean: Overall activity level
      * Std: Variability (neuron firing consistently vs erratically?)
      * Max: Peak response (how strong was the maximum activation?)
      * Early: Initial response (first 1/3 of stimulus)
      * Late: Sustained response (last 1/3 of stimulus)
    - Features: [neuron_1_mean, neuron_1_std, ..., neuron_1_late, neuron_2_mean, ...]
    - Captures richer dynamics but increases dimensionality

    1. Keep raw features
    2. Split into train/test
    3. Compute mean/std on training data ONLY
    4. Z-score train set using train statistics
    5. Z-score test set using train statistics
    6. Train model

    Parameters:
    rs : ROI response series object
        Contains neural data and timestamps
    timestamps : array
        Time of each recording frame
    clip_starts, clip_stops, clip_types : arrays
        Stimulus presentation timing and labels
    neuron_list : list
        Which neurons to include (after feature selection)
    target_conditions : list
        Which stimulus types to include (e.g., ['Cinematic', 'Rendered'])
    window_duration : float or None
        How long to analyze after stimulus onset (None = full clip)
    use_rich_features : bool
        Whether to extract multiple temporal features
    """

    # Create a mapping from condition name to integer label
    # Machine learning models need numeric labels, not strings
    # Example: {'Cinematic': 0, 'Rendered': 1, 'sports1m': 2}
    condition_names = {cond: i for i, cond in enumerate(target_conditions)}

    # Initialize lists to store extracted trials and their labels
    all_trials = []  # Will hold feature vectors for each trial
    all_labels = []  # Will hold corresponding class labels

    # Loop through each stimulus presentation
    for i in range(len(clip_starts)):
        stim_type = clip_types[i]  # What type of clip was shown

        # Skip if this clip type is not in our target conditions
        # Example: If we only care about 'Cinematic' and 'Rendered',
        #          skip 'monet2' clips
        if stim_type not in target_conditions:
            continue

        # Define the time window we're analyzing
        start_time = clip_starts[i]  # When stimulus appeared

        if window_duration is None:
            # Use the full duration until clip ends
            stop_time = clip_stops[i]
        else:
            # Use only a fixed window after onset
            # Example: Only analyze first 1 second after stimulus
            stop_time = start_time + window_duration

        # Convert times to array indices
        # searchsorted finds where to insert value to keep array sorted
        # This gives us the index of the closest timestamp
        start_idx = np.searchsorted(timestamps, start_time)
        stop_idx = np.searchsorted(timestamps, stop_time)

        # Extract neural activity during this time window
        # rs.data[start_idx:stop_idx, neuron_list] gives us:
        #   Rows: Time points during this trial
        #   Cols: Selected neurons
        # We convert to numpy array for easier manipulation
        neural_chunk = np.array(rs.data[start_idx:stop_idx, neuron_list])

        if use_rich_features:

            features = []  # Will concatenate all features

            # FEATURE 1: Mean activity across time
            # Shape: [num_neurons]
            # Captures overall response magnitude
            features.append(np.mean(neural_chunk, axis=0))

            # FEATURE 2: Standard deviation across time
            # Shape: [num_neurons]
            # Captures response variability
            # High std = neuron response fluctuates
            # Low std = neuron response steady
            features.append(np.std(neural_chunk, axis=0))

            # FEATURE 3: Maximum activity across time
            # Shape: [num_neurons]
            # Captures peak response strength
            features.append(np.max(neural_chunk, axis=0))

            # FEATURE 4 & 5: Early vs Late response
            # Only if we have enough timepoints (at least 20)
            if neural_chunk.shape[0] >= 20:
                # Compute early response (first ~1/3 of stimulus)
                n_early = min(10, neural_chunk.shape[0] // 3)
                # Compute late response (last ~1/3 of stimulus)
                n_late = min(10, neural_chunk.shape[0] // 3)

                # Average activity in early window
                # Some neurons respond transiently (early only)
                features.append(np.mean(neural_chunk[:n_early], axis=0))

                # Average activity in late window
                # Some neurons have sustained responses
                features.append(np.mean(neural_chunk[-n_late:], axis=0))

            # Concatenate all features into one vector
            # If we have 200 neurons and 5 feature types:
            # Result shape: [1000] (200 neurons × 5 features)
            neural_features = np.concatenate(features)
        else:
            # SIMPLE MODE: Just use mean
            # Shape: [num_neurons]
            # axis=0 means average across time (rows)
            neural_features = np.mean(neural_chunk, axis=0)

        # Store this trial's features and label
        all_trials.append(neural_features)
        all_labels.append(condition_names[stim_type])

    # Convert lists to numpy arrays
    # X: Feature matrix [num_trials × num_features]
    # y: Label vector [num_trials]
    X = np.array(all_trials)
    y = np.array(all_labels)

    return X, y, condition_names


# ============================================================================
# Gridsearch
# ============================================================================

def grid_search_decoder(X, y, n_folds=5):
    """
    Use GridSearchCV with Pipeline to find best hyperparameters.
    Pipeline ensures z-scoring is fit only on training data in each fold.
    """

    # Create a pipeline: StandardScaler + LogisticRegression
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(random_state=42))
    ])

    # Define parameter grid with VALID solver-penalty combinations only
    param_grid = [
        # L2 penalty
        {
            'classifier__C': [0.001, 0.01, 0.1, 1.0, 10, 100],
            'classifier__penalty': ['l2'],
            'classifier__solver': ['lbfgs', 'liblinear', 'saga'],
            'classifier__class_weight': [None, 'balanced'],
            'classifier__max_iter': [1000, 2000]
        },
        # L1 penalty
        {
            'classifier__C': [0.001, 0.01, 0.1, 1.0, 10, 100],
            'classifier__penalty': ['l1'],
            'classifier__solver': ['liblinear', 'saga'],
            'classifier__class_weight': [None, 'balanced'],
            'classifier__max_iter': [1000, 2000]
        },
        # No penalty
        {
            'classifier__penalty': [None],
            'classifier__solver': ['lbfgs', 'saga'],
            'classifier__class_weight': [None, 'balanced'],
            'classifier__max_iter': [1000, 2000]
        },
        # ElasticNet
        {
            'classifier__C': [0.001, 0.01, 0.1, 1.0, 10, 100],
            'classifier__penalty': ['elasticnet'],
            'classifier__solver': ['saga'],
            'classifier__l1_ratio': [0.5],
            'classifier__class_weight': [None, 'balanced'],
            'classifier__max_iter': [2000]
        }
    ]

    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1,
        refit=True
    )

    grid_search.fit(X, y)

    return grid_search.best_estimator_, grid_search.best_params_, grid_search.best_score_


def evaluate_best_model(best_pipeline, X, y, n_folds=5):
    """Evaluate the best model with cross-validation."""

    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    fold_accuracies = []

    for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        from sklearn.base import clone
        pipeline = clone(best_pipeline)
        pipeline.fit(X_train, y_train)

        y_pred = pipeline.predict(X_test)
        accuracy = np.mean(y_test == y_pred)
        fold_accuracies.append(accuracy)

    return fold_accuracies


# ============================================================================
# Analysis
# ============================================================================

print("Neural Decoding Analysis with GridSearchCV\n")

# Load data
nwb = connect_to_data()
rs = get_neural_data(nwb)
clip_starts, clip_stops, clip_types = get_stimulus_info(nwb)

timestamps = np.array(rs.timestamps[:100000])
target_conditions = ['Cinematic', 'Rendered', 'sports1m']

n_neurons = 200

# Select BEST neurons instead of arbitrary first 200
neuron_list = select_best_neurons(
    rs, timestamps, clip_starts, clip_stops, clip_types,
    target_conditions, n_select=n_neurons
)

# Try with rich temporal features
use_rich_features = False  # Set to False to use only mean (original)

X_full, y_full, condition_names = extract_neural_features(
    rs, timestamps, clip_starts, clip_stops, clip_types,
    neuron_list, target_conditions, window_duration=None,
    use_rich_features=use_rich_features
)


# ============================================================================
# Grid Search
# ============================================================================

n_folds = 5
best_pipeline, best_params, best_score = grid_search_decoder(X_full, y_full, n_folds=n_folds)

fold_accs = evaluate_best_model(best_pipeline, X_full, y_full, n_folds=n_folds)

# ============================================================================
# Results
# ============================================================================

print("\n" + "="*70)
print("Results:")
print("="*70)

# Clean up parameter names
params_clean = {k.replace('classifier__', ''): v for k, v in best_params.items()}

print(f"\nAccuracy: {np.mean(fold_accs)*100:.2f}% ± {np.std(fold_accs)*100:.2f}%")
print(f"\nBest parameters:")
for param, value in params_clean.items():
    print(f"  {param}: {value}")
print("\n" + "="*70)


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.2/179.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.3/85.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 339.0/339.0 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.9/119.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.1/753.1 kB 49.4 MB/s eta 0:00:00
   ━━━━━━